# Ekstraksi Fitur  
*Dari Time Series Polutan NO₂ ke 68 Fitur TSFEL*

> Di tahap **Ekstraksi Fitur**, kita mengubah deret waktu harian konsentrasi **NO₂** (dari Sentinel‑5P) menjadi **68 fitur numerik** menggunakan library **TSFEL** (Time Series Feature Extraction Library).  
>  
> Fitur‑fitur ini nantinya bisa dipakai untuk:
> - analisis statistik lanjutan,
> - clustering kecamatan berdasarkan pola polusi,
> - atau建模 machine learning (mis. memprediksi tingkat polusi).

Prosesnya mengikuti alur:

1. Ambil data NO₂ untuk **satu kecamatan** (AOI tertentu).  
2. Lakukan **preprocessing**:  
   - konversi ke numerik,  
   - deteksi outlier,  
   - imputasi missing value.  
3. Ekstrak **68 fitur** menggunakan **TSFEL**.  
4. Simpan hasil ekstraksi fitur ke dalam **tabel di database Aiven PostgreSQL**.

---

## 1. Pengambilan Data per Kecamatan

### 1.1 Menentukan AOI Kecamatan

Setiap kelompok mengambil data untuk **satu kecamatan** tertentu. Contoh AOI yang digunakan di notebook ini adalah untuk kecamatan dengan bounding box berikut (sesuaikan dengan kecamatan kamu):

```python
aoi = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {},
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [114.2666, -8.523975],
                        [114.2666, -8.36999],
                        [114.367188, -8.36999],
                        [114.367188, -8.523975],
                        [114.2666, -8.523975]
                    ]
                ]
            }
        }
    ]
}
```

> Catatan: Ganti koordinat ini sesuai AOI kecamatan kamu (bisa ditentukan lewat geojson.io seperti di tahap sebelumnya).

### 1.2 Mengambil Data NO₂ via openEO

Periode pengamatan yang digunakan:

- **31 Agustus 2025 – 31 Agustus 2026**

Berikut contoh kode untuk mengambil data **NO₂** dengan openEO:

```python
import openeo

# Koneksi ke openEO Copernicus
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

# Definisi AOI (sesuaikan dengan kecamatan kamu)
aoi = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {},
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [114.2666, -8.523975],
                        [114.2666, -8.36999],
                        [114.367188, -8.36999],
                        [114.367188, -8.523975],
                        [114.2666, -8.523975]
                    ]
                ]
            }
        }
    ]
}

# Load koleksi Sentinel-5P untuk NO2
s5NO2 = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=["2025-08-31", "2026-08-31"],
    spatial_extent={
        "west": 114.2666,
        "south": -8.523975,
        "east": 114.367188,
        "north": -8.36999
    },
    bands=["NO2"],
)

# Agregasi harian
s5p_NO2_daily = s5NO2.aggregate_temporal_period(
    reducer="mean",
    period="day"
)

# Agregasi spasial berdasarkan AOI
s5p_NO2_aoi = s5p_NO2_daily.aggregate_spatial(
    reducer="mean",
    geometries=aoi
)

# Simpan hasil sebagai CSV
result_NO2 = s5p_NO2_aoi.save_result(format="CSV")

# Buat dan jalankan job
job_NO2 = result_NO2.create_job(title="s5p_NO2_timeseries")
job_NO2.start_and_wait()

# Download CSV
job_NO2.get_results().download_files("NO2_Kamal")
```

> Ganti `"NO2_Kamal"` dengan nama folder sesuai kecamatan kamu (mis. `"NO2_Muncar"`).

Hasilnya adalah file CSV dengan struktur kira‑kira:

- `date` → tanggal  
- `NO2` → nilai konsentrasi NO₂ (rata‑rata harian di AOI kecamatan).

---

## 2. Preprocessing Sebelum Ekstraksi Fitur

Sebelum masuk ke ekstraksi fitur, data harus “dibersihkan” dulu:

1. Pastikan hanya nilai **numerik** yang dipakai.  
2. Deteksi dan **tandai outlier** (diubah jadi NaN).  
3. Lakukan **imputasi missing value** sehingga tidak ada lagi NaN.  

### 2.1 Memuat Data dan Konversi ke Numerik

```python
import pandas as pd
import numpy as np

# Muat data CSV hasil openEO
df = pd.read_csv('NO2_Kamal/timeseries.csv')

# Pastikan kolom date dalam format datetime
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

target_pollutant = 'NO2'

# Paksa kolom target jadi numerik; nilai yang gagal dikonversi -> NaN
df[target_pollutant] = pd.to_numeric(df[target_pollutant], errors='coerce')

n_missing_before = df[target_pollutant].isna().sum()
print(f"Jumlah nilai non-numerik/kosong yang dikonversi jadi NaN: {n_missing_before}")
```

---

### 2.2 Deteksi dan Penanganan Outlier

Outlier dideteksi menggunakan metode **IQR** (Interquartile Range).

```python
# Hitung Q1, Q3, IQR
Q1 = df[target_pollutant].quantile(0.25)
Q3 = df[target_pollutant].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Tandai outlier dan ubah jadi NaN
outlier_mask = (df[target_pollutant] < lower_bound) | (df[target_pollutant] > upper_bound)
n_outlier = outlier_mask.sum()
print(f"Jumlah outlier yang diubah jadi NaN: {n_outlier}")

df.loc[outlier_mask, target_pollutant] = np.nan
```

Setelah langkah ini, data mungkin punya lebih banyak NaN (karena outlier “dibuang” dan dianggap missing).

---

### 2.3 Imputasi Missing Value

Untuk memastikan data “sempurna” (tanpa missing value), kita lakukan imputasi dengan:

- **interpolasi waktu** (`method='time'`),  
- dilanjutkan **forward fill** dan **backward fill** untuk mengisi kemungkinan NaN di ujung seri.

```python
# Set date sebagai index untuk interpolasi berbasis waktu
df_clean = df.set_index('date').interpolate(method='time').ffill().bfill()

# Cek apakah masih ada NaN
remaining_missing = df_clean[target_pollutant].isna().sum()
print(f"Sisa missing value setelah imputasi: {remaining_missing}")
```

Sekarang `df_clean[target_pollutant]` sudah tidak memiliki missing value dan siap untuk ekstraksi fitur.

---

## 3. Ekstraksi 68 Fitur dengan TSFEL

Library yang digunakan: **TSFEL** (Time Series Feature Extraction Library).  
Target: menghasilkan **68 fitur** untuk deret waktu NO₂.

### 3.1 Persiapan Library dan Daftar Fitur

```python
import tsfel.feature_extraction.features as tsfel_features
import inspect

# Daftar PERSIS 68 fitur yang diminta
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print("Jumlah fitur yang diminta:", len(FEATURE_LIST))
```

---

### 3.2 Fungsi Ekstraksi Fitur per Nama

Karena TSFEL punya banyak fungsi dengan signature berbeda (ada yang butuh `fs`, ada yang tidak), kita buat helper:

```python
def to_scalar(result):
    """Konversi hasil fungsi TSFEL ke skalar (float)."""
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)


def extract_one(fn_name, signal, fs):
    """Ekstrak satu fitur berdasarkan nama fungsi."""
    try:
        fn = getattr(tsfel_features, fn_name)
    except AttributeError:
        raise ValueError(f"Fungsi {fn_name} tidak ditemukan di tsfel_features")

    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)
```

---

### 3.3 Ekstraksi 68 Fitur untuk NO₂

Siapkan sinyal 1D dan sampling frequency:

```python
# Sampling frequency: 1 observasi per hari
fs = 1

# Sinyal 1D (NO2) sebagai numpy array float
signal_1d = df_clean[target_pollutant].astype(float).values
```

Lakukan ekstraksi:

```python
row = {}
for fn_name in FEATURE_LIST:
    row[fn_name] = extract_one(fn_name, signal_1d, fs)

extracted_features_final = pd.DataFrame([row])

print(f"Berhasil! Jumlah fitur yang dihasilkan untuk {target_pollutant}: {extracted_features_final.shape}")[1]
```

Simpan hasil ekstraksi fitur ke CSV:

```python
extracted_features_final.to_csv(f'{target_pollutant}_Kamal_TSFEL.csv', index=False)
```

File `NO2_Kamal_TSFEL.csv` ini berisi **satu baris** dengan **68 kolom fitur** untuk kecamatan tersebut.

---

## 4. Mengelompokkan Fitur Berdasarkan Domain

TSFEL membagi fitur ke dalam beberapa domain, antara lain:

- **Statistical features** (statistik waktu):  
  `calc_mean`, `calc_std`, `calc_var`, `skewness`, `kurtosis`, `rms`, `zero_cross`, dll.  
- **Temporal features**:  
  `autocorr`, `entropy`, `hurst_exponent`, `higuchi_fractal_dimension`, `lempel_ziv`, dll.  
- **Spectral features**:  
  `spectral_centroid`, `spectral_skewness`, `spectral_kurtosis`, `spectral_slope`, `power_bandwidth`, `max_frequency`, dll.

Untuk keperluan laporan, kamu bisa membuat tabel seperti:

| No | Nama Fitur                  | Domain     |
|----|-----------------------------|------------|
| 1  | calc_mean                   | Statistical|
| 2  | calc_std                    | Statistical|
| 3  | skewness                    | Statistical|
| 4  | kurtosis                    | Statistical|
| 5  | autocorr                    | Temporal   |
| 6  | entropy                     | Temporal   |
| 7  | hurst_exponent              | Temporal   |
| 8  | spectral_centroid           | Spectral   |
| 9  | spectral_skewness           | Spectral   |
|10  | spectral_kurtosis           | Spectral   |
|…   | …                           | …          |

(Kamu bisa melengkapi tabel ini berdasarkan dokumentasi TSFEL atau dengan memeriksa nama fungsi satu per satu.)

---

## 5. Menyimpan Hasil Ekstraksi Fitur ke Aiven PostgreSQL

Setelah file CSV fitur jadi, langkah terakhir adalah memasukkan hasil ekstraksi fitur ke dalam **tabel di database Aiven PostgreSQL** yang sudah kamu gunakan di tahap Cloud Data.

### 5.1 Membuat Tabel Fitur di PostgreSQL

Misalnya, kita buat tabel `no2_features_kecamatan`:

```sql
CREATE TABLE no2_features_kecamatan (
    id SERIAL PRIMARY KEY,
    kecamatan_name TEXT,
    abs_energy DOUBLE PRECISION,
    auc DOUBLE PRECISION,
    autocorr DOUBLE PRECISION,
    average_power DOUBLE PRECISION,
    calc_centroid DOUBLE PRECISION,
    calc_max DOUBLE PRECISION,
    calc_mean DOUBLE PRECISION,
    calc_median DOUBLE PRECISION,
    calc_min DOUBLE PRECISION,
    calc_std DOUBLE PRECISION,
    calc_var DOUBLE PRECISION,
    dfa DOUBLE PRECISION,
    distance DOUBLE PRECISION,
    ecdf DOUBLE PRECISION,
    ecdf_percentile DOUBLE PRECISION,
    ecdf_percentile_count DOUBLE PRECISION,
    ecdf_slope DOUBLE PRECISION,
    entropy DOUBLE PRECISION,
    fundamental_frequency DOUBLE PRECISION,
    higuchi_fractal_dimension DOUBLE PRECISION,
    hist_mode DOUBLE PRECISION,
    human_range_energy DOUBLE PRECISION,
    hurst_exponent DOUBLE PRECISION,
    interq_range DOUBLE PRECISION,
    kurtosis DOUBLE PRECISION,
    lempel_ziv DOUBLE PRECISION,
    lpcc DOUBLE PRECISION,
    max_frequency DOUBLE PRECISION,
    max_power_spectrum DOUBLE PRECISION,
    maximum_fractal_length DOUBLE PRECISION,
    mean_abs_deviation DOUBLE PRECISION,
    mean_abs_diff DOUBLE PRECISION,
    mean_diff DOUBLE PRECISION,
    median_abs_deviation DOUBLE PRECISION,
    median_abs_diff DOUBLE PRECISION,
    median_diff DOUBLE PRECISION,
    median_frequency DOUBLE PRECISION,
    mfcc DOUBLE PRECISION,
    mse DOUBLE PRECISION,
    negative_turning DOUBLE PRECISION,
    neighbourhood_peaks DOUBLE PRECISION,
    petrosian_fractal_dimension DOUBLE PRECISION,
    pk_pk_distance DOUBLE PRECISION,
    positive_turning DOUBLE PRECISION,
    power_bandwidth DOUBLE PRECISION,
    rms DOUBLE PRECISION,
    skewness DOUBLE PRECISION,
    slope DOUBLE PRECISION,
    spectral_centroid DOUBLE PRECISION,
    spectral_decrease DOUBLE PRECISION,
    spectral_distance DOUBLE PRECISION,
    spectral_entropy DOUBLE PRECISION,
    spectral_kurtosis DOUBLE PRECISION,
    spectral_positive_turning DOUBLE PRECISION,
    spectral_roll_off DOUBLE PRECISION,
    spectral_roll_on DOUBLE PRECISION,
    spectral_skewness DOUBLE PRECISION,
    spectral_slope DOUBLE PRECISION,
    spectral_spread DOUBLE PRECISION,
    spectral_variation DOUBLE PRECISION,
    spectrogram_mean_coeff DOUBLE PRECISION,
    sum_abs_diff DOUBLE PRECISION,
    wavelet_abs_mean DOUBLE PRECISION,
    wavelet_energy DOUBLE PRECISION,
    wavelet_entropy DOUBLE PRECISION,
    wavelet_std DOUBLE PRECISION,
    wavelet_var DOUBLE PRECISION,
    zero_cross DOUBLE PRECISION
);
```

> Kamu bisa membuat script SQL ini sekali, lalu dipakai untuk semua kecamatan (tambah baris baru per kecamatan).

### 5.2 Mengimpor CSV Fitur ke PostgreSQL (via DBeaver)

1. Buka **DBeaver**, pastikan koneksi ke Aiven PostgreSQL sudah aktif.  
2. Navigasi ke:  
   `Databases > defaultdb > Schemas > public > Tables`.  
3. Klik kanan pada tabel `no2_features_kecamatan`, pilih **Import Data...**.  
4. Pilih file CSV `NO2_Kamal_TSFEL.csv`.  
5. Pastikan mapping kolom sesuai (nama kolom di CSV = nama kolom di tabel).  
6. Tambahkan kolom `kecamatan_name` secara manual (mis. isi `"Kamal"`) setelah import, atau tambahkan kolom ini di CSV sebelum diimpor.

Alternatif: insert manual via SQL:

```sql
INSERT INTO no2_features_kecamatan (
    kecamatan_name,
    abs_energy,
    auc,
    autocorr,
    -- ... lanjutkan sampai zero_cross
    zero_cross
)
VALUES (
    'Kamal',
    0.123,  -- abs_energy
    0.456,  -- auc
    0.789,  -- autocorr
    -- ...
    0.012   -- zero_cross
);
```

Nilai‑nilai ini bisa kamu ambil dari `extracted_features_final` di Python:

```python
row = extracted_features_final.iloc
print(row['abs_energy'], row['auc'], row['autocorr'], ...)
```

---

## 6. Ringkasan Tahapan Ekstraksi Fitur

Secara ringkas, alur kerja di tahap ini adalah:

1. **Ambil data NO₂** untuk satu kecamatan (AOI tertentu) dari Sentinel‑5P via openEO (periode 31‑08‑2025 s.d. 31‑08‑2026).  
2. **Preprocessing**:  
   - konversi ke numerik,  
   - deteksi outlier dengan IQR dan ubah jadi NaN,  
   - imputasi missing value (interpolasi + ffill/bfill).  
3. **Ekstraksi 68 fitur** menggunakan **TSFEL** untuk deret waktu NO₂.  
4. Simpan hasil fitur ke CSV, lalu **unggah ke tabel PostgreSQL di Aiven**.  
5. Kelompokkan fitur berdasarkan domain (**statistical**, **temporal**, **spectral**) untuk keperluan laporan dan interpretasi.

Setelah semua selesai, kamu sudah punya:

- satu baris fitur per kecamatan di database cloud,  
- siap untuk analisis lanjutan (mis. clustering kecamatan berdasarkan pola polusi, atau regresi terhadap variabel lain).